# v0.3 Step 1 — Yeast8 손에 익히기

**목표**: Yeast8을 직접 한 번 굴려본다. 이해 못 해도 괜찮다. 끝까지 실행하고 *어디서 막혔는지* 기록하는 게 더 중요하다.

**Yeast8이 뭐냐?** Saccharomyces cerevisiae의 *전체 게놈 규모* 대사 모델. 4000개 이상 반응, 2800개 대사물질, 1100개 유전자. 너의 v0.1 ABM이 ~10개 변수였던 것과 *규모가 다르다*. 학계 표준이고, *진짜로 쓰이는* 도구.

**이 노트북에서 하는 것:**
1. Yeast8 모델 다운로드 + 로드
2. 기본 FBA (성장률 예측)
3. 간단한 dFBA (시간 변화 추적) — batch fermentation
4. *어디서 막혔는지 학습 일지 쓰기*

**시간**: 30-45분. 처음이면 모르는 게 많을 거고, 그게 정상이다.

**중요**: 이 노트북은 *완성된 도구*가 아니다. *학습 도구*다. 셀마다 "왜 이걸 하는지" 주석을 달아뒀으니 천천히 읽어라.

## 0. 환경 설치 (Colab에서 1번만)

Colab은 기본적으로 cobrapy를 안 깔아둠. 한 번 깔아야 됨. 약 30초.

In [ ]:
!pip install cobra --quiet

## 1. Yeast8 SBML 모델 다운로드

Yeast8은 GitHub의 yeast-GEM repo에 있다. 우리는 SBML(.xml) 파일 하나만 받으면 된다. 약 11MB.

**SBML?** Systems Biology Markup Language. 대사 모델의 표준 형식. 너가 직접 볼 필요는 없고, cobrapy가 알아서 읽어준다.

In [ ]:
import urllib.request
import os

url = 'https://raw.githubusercontent.com/SysBioChalmers/yeast-GEM/main/model/yeast-GEM.xml'
local_path = 'yeast-GEM.xml'

if not os.path.exists(local_path):
    print('Downloading...')
    urllib.request.urlretrieve(url, local_path)

size_mb = os.path.getsize(local_path) / 1024 / 1024
print(f'OK: {local_path} ({size_mb:.1f} MB)')

## 2. 모델 로드

cobrapy가 SBML을 읽어서 Python 객체로 만들어준다. 처음에는 15-30초 걸린다 — 4000개 반응 다 파싱하니까.

In [ ]:
import cobra

model = cobra.io.read_sbml_model(local_path)

print(f'Reactions:   {len(model.reactions):>5}')
print(f'Metabolites: {len(model.metabolites):>5}')
print(f'Genes:       {len(model.genes):>5}')
print()
print(f'Objective: {model.objective.expression}')

**여기까지 됐으면 너는 시스템생명공학에서 *진짜로* 쓰는 도구를 손에 쥔 상태다.**

이 모델 하나에 효모의 거의 모든 알려진 대사 반응이 들어있다. 위에서 출력된 `r_2111`이 'biomass formation' 반응 — 즉 모델의 *목표*가 "biomass를 최대화하는 flux 분포를 찾기"로 설정돼 있다.

## 3. 첫 FBA — 한 번 풀어보기

**FBA (Flux Balance Analysis)** = "이 대사 네트워크가 *정상상태*에서 어떤 flux 분포를 가지면 biomass가 최대가 되나?"를 선형계획법으로 푸는 것.

지금 모델은 *기본 배지* 조건에 설정돼 있다. 그대로 풀어보자.

In [ ]:
solution = model.optimize()

print(f'Status: {solution.status}')
print(f'Growth rate (mu): {solution.objective_value:.4f} 1/hr')
print()
print('FBA가 푼 정상상태에서 in/out flux 요약:')
print(model.summary())

**무엇이 보였나?**
- `mu` (성장률) ≈ 0.086 /hr — 실제 효모 chemostat 데이터와 같은 범위. 모델이 *생물학적으로 말이 되는* 답을 내놨다.
- `model.summary()`는 글루코스/암모니아/산소가 들어오고 biomass/CO2/water가 나가는 걸 보여준다. 효모 대사의 *전체 모습*이 한 번에 나온다.

너의 v0.1 ABM은 이런 거 *못 했다*. ABM은 "개별 cell의 상태"를 봤지만, 어떤 *분자가 어디로 흐르는지*는 없었다. Yeast8은 그 분자 수준 답을 준다.

## 4. 핵심 exchange reactions 찾기

우리가 fed-batch를 시뮬레이션하려면 *바깥 세계와 주고받는* 반응들을 알아야 한다. SBML 모델에서 이런 반응을 'exchange reaction'이라고 부르고 보통 `EX_*` 또는 (Yeast8의 경우) `r_*` 형식으로 이름이 붙어있다.

Yeast8에서 자주 쓰는 exchange reactions:

In [ ]:
important_exchanges = {
    'r_1714': 'D-glucose exchange (탄소원, in)',
    'r_1654': 'ammonium exchange (질소원, in)',
    'r_1992': 'oxygen exchange (in)',
    'r_1672': 'CO2 exchange (out)',
    'r_1761': 'ethanol exchange (out)',
    'r_1808': 'glycerol exchange (out)',
    'r_1634': 'acetate exchange (out)',
    'r_2111': 'biomass pseudoreaction (objective)',
}

for rxn_id, desc in important_exchanges.items():
    try:
        rxn = model.reactions.get_by_id(rxn_id)
        print(f'{rxn_id}  bounds=[{rxn.lower_bound:>6.1f}, {rxn.upper_bound:>6.1f}]  {desc}')
    except KeyError:
        print(f'{rxn_id}  (not found in this version of yeast-GEM)')

**해석:**
- 음수 lower_bound = uptake (외부 → cell). 예: 글루코스가 lower=-1.0이면 "시간당 최대 1mmol/gDW 흡수 가능".
- 양수 upper_bound = secretion (cell → 외부).

**이게 핵심**: fed-batch에서 *우리가 조절하는 것*은 바로 이 `lower_bound`다. 글루코스 feed rate를 바꿔주려면 `r_1714.lower_bound`를 매 시간 step마다 업데이트하면 된다.

## 5. dFBA — 시간 따라 시뮬레이션

**Dynamic FBA**: 각 시간 step마다 (1) 현재 글루코스 양 → uptake 가능량 계산, (2) FBA 풀어서 mu와 byproduct flux 얻기, (3) biomass/글루코스/byproduct 다음 step 업데이트, (4) 반복.

이게 너의 v0.1 ABM이 했던 것과 *정확히 같은 형태*다. 다만 ABM은 개별 cell이고, Yeast8 dFBA는 *집단 평균 대사*다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 초기 조건 (KCCM 12638 논문의 batch phase 모사) ---
biomass_0 = 0.1   # g/L
glucose_0 = 20.0  # g/L
dt = 0.5          # 0.5시간씩 진행
n_steps = 40      # 총 20시간

# --- Monod kinetics 파라미터 ---
Ks = 0.5    # g/L, glucose 반포화 상수
vmax = 10   # mmol/gDW/hr, 최대 uptake rate (효모 일반값)

# --- 시뮬레이션 루프 ---
times = [0.0]
biomasses = [biomass_0]
glucoses = [glucose_0]
ethanols = [0.0]
growth_rates = [0.0]

biomass = biomass_0
glucose = glucose_0
ethanol = 0.0

glc_exch = model.reactions.get_by_id('r_1714')
etoh_exch = model.reactions.get_by_id('r_1761')

for step in range(n_steps):
    # 현재 글루코스 농도 → Monod로 uptake 결정
    v_uptake = vmax * glucose / (Ks + glucose) if glucose > 1e-6 else 0
    glc_exch.lower_bound = -v_uptake   # 음수 = uptake 방향

    # FBA 풀기
    sol = model.optimize()
    if sol.status != 'optimal' or sol.objective_value is None:
        print(f't={step*dt:.1f}h: FBA infeasible. Likely glucose depleted.')
        break

    mu = sol.objective_value
    v_etoh = sol.fluxes['r_1761']

    # 다음 step 업데이트 (단순 Euler)
    # 단위 변환: mmol/gDW/hr * gDW/L * g/mmol * hr = g/L
    glucose = max(0, glucose - v_uptake * biomass * 0.18 * dt)  # 0.18 = MW(glucose)/1000
    biomass = biomass * np.exp(mu * dt)
    ethanol = ethanol + v_etoh * biomass * 0.046 * dt

    times.append((step+1) * dt)
    biomasses.append(biomass)
    glucoses.append(glucose)
    ethanols.append(ethanol)
    growth_rates.append(mu)

print(f'\nSimulation ended at t={times[-1]:.1f}h')
print(f'Final biomass: {biomasses[-1]:.2f} g/L')
print(f'Final glucose: {glucoses[-1]:.2f} g/L')

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(times, biomasses, 'b-', lw=2)
axes[0].set_xlabel('time (h)'); axes[0].set_ylabel('biomass (g/L)')
axes[0].set_title('Biomass'); axes[0].grid(alpha=0.3)

axes[1].plot(times, glucoses, 'r-', lw=2)
axes[1].set_xlabel('time (h)'); axes[1].set_ylabel('glucose (g/L)')
axes[1].set_title('Glucose'); axes[1].grid(alpha=0.3)

axes[2].plot(times, growth_rates, 'g-', lw=2)
axes[2].set_xlabel('time (h)'); axes[2].set_ylabel('mu (1/h)')
axes[2].set_title('Growth rate (predicted)'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 학습 일지 — 솔직히 적기

이 노트북을 처음부터 끝까지 실행하면서 *너가 *모르는 것*과 *놀란 것*을 솔직히 기록해라. 이게 너가 사수형이나 효모 교수님 만났을 때 *질문할 것*들이다.

예시:
- "mu가 0.086 1/hr이라는데 KCCM 12638 논문의 실제 mu는 얼마였지?"
- "내 dFBA에서 ethanol이 0이 나왔는데, KCCM 데이터는 ethanol이 11 g/L 정도 됐다. Yeast8이 Crabtree effect를 안 모델링하나? 산소 조건 바꿔야 하나?"
- "실제 fed-batch에서는 batch phase 다음 글루코스를 천천히 feed하는데, 이걸 어떻게 표현하지? glc_exch.lower_bound를 매 step 다르게 주면 되나?"
- "`r_2111` (biomass pseudoreaction)이 *진짜 효모 biomass*와 같은 의미인가? 단위는?"
- "KCCM 12638 같은 *특정 균주*를 모델링하려면 Yeast8을 그대로 쓰면 되나, 아니면 균주별 수정이 필요한가?"

**이 의문들이 곧 v0.4의 todo list가 된다.**

## 다음 단계 (이 노트북 끝나면)

**Step 2 (다음 노트북)**: Yeast8 dFBA를 N번 반복하면서 *변동성 주입* — 너의 v0.1 ABM이 했던 Monte Carlo를 Yeast8 위에서 재현.

**Step 3**: pulsed feeding vs constant feeding을 Yeast8 dFBA로 비교 → 너의 호르메시스 가설을 *검증된 모델 위에서* 테스트.

**Step 4**: black swan event (toxin pulse) 주입 → robustness 비교.

지금은 Step 1만 — *돌아간다는 사실*만 확인하면 된다.